In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import MinMaxScaler

# ======= Load data =======
# Load patient event logs and activity definitions
log = pd.read_csv("../../data/raw_data/general_check_up/general_check_up.csv")
with open("../../data/raw_data/general_check_up/general_check_up_activities_info.json", "r") as f:
    ACTIVITIES = json.load(f)

# ======= Time range setup =======
# Determine the full range of timestamps in the log
max_timestamp = round(log['end_timestamp'].max())
min_timestamp = round(log['start_timestamp'].min())
list_timestamp = list(range(min_timestamp, max_timestamp + 1))

# ======= Initialize activity statistics =======
# Prepare a dictionary to store number of patients at each activity over time
activity_names = list(ACTIVITIES.keys())
case_statistics = {act: [] for act in activity_names}

# Count number of patients at each activity at every timestamp
for timestamp in list_timestamp:
    # Select patients active at this timestamp
    df_timestamp = log[(log['start_timestamp'] <= timestamp) & (log['end_timestamp'] >= timestamp)]
    count_by_activity = df_timestamp['activity_name'].value_counts().to_dict()
    
    # Fill counts for all activities
    for act in activity_names:
        case_statistics[act].append(count_by_activity.get(act, 0))

# Convert to DataFrame with timestamp as index
df_case_statistics = pd.DataFrame(case_statistics, index=list_timestamp)

# ======= Generate prefix sequences =======
def generate_prefix(row, log):
    """
    Generate the prefix (sequence of activities) for a given row in the log.
    Prefix includes all activities of the patient up to and including the current activity.
    """
    case_id = row['patient_id']
    log_of_case = log[log['patient_id'] == case_id].sort_values(by='start_timestamp')
    activity = row['activity_name']
    
    prefix = []
    for act in log_of_case['activity_name']:
        prefix.append(act)
        if act == activity:
            break
    return prefix

# Apply prefix generation to all rows
log['prefix'] = log.apply(lambda row: generate_prefix(row, log), axis=1)

# ======= Estimate remaining time at each activity =======
# Formula: remaining_time = number_of_cases_at_node * mean_service_time / number_of_staff
df_remaining_time = pd.DataFrame(index=df_case_statistics.index, columns=df_case_statistics.columns)

for act in df_case_statistics.columns:
    act_info = ACTIVITIES[act]
    mean_time = act_info['mean_time']
    n_resources = act_info['staff']
    
    # If mean_time is a dict (e.g., different times for sub-tasks), take the average
    if isinstance(mean_time, dict):
        mean_time = np.mean(list(mean_time.values()))
    
    # Compute estimated remaining time at each timestamp
    df_remaining_time[act] = df_case_statistics[act] * mean_time / n_resources

# Convert remaining time info to dict and vector for RL input
df_remaining_time['env_info'] = df_remaining_time.apply(lambda row: row.to_dict(), axis=1)
df_remaining_time['env_info_vector'] = df_remaining_time['env_info'].apply(lambda x: list(x.values()))

# ======= Merge environment info into log =======
log_merged = log.copy()
log_merged['merge_timestep'] = np.floor(log_merged['start_timestamp']).astype(int)

# Reset index for merging
df_remaining_time = df_remaining_time.reset_index().rename(columns={'index': 'timestep'})

# Merge remaining time info into log by timestep
log_merged = log_merged.merge(
    df_remaining_time[['timestep', 'env_info', 'env_info_vector']],
    left_on='merge_timestep',
    right_on='timestep',
    how='left'
)

# ======= Normalize features =======
# Normalize waiting time
scaler = MinMaxScaler()
log_merged['waiting_time_norm'] = scaler.fit_transform(log_merged[['waiting_time']])

# Normalize environment vector for each row
def scale_row_manual(vector):
    vector = np.array(vector, dtype=float)
    min_val = vector.min()
    max_val = vector.max()
    if max_val - min_val == 0:
        return np.zeros_like(vector).tolist()  # hoặc np.ones_like(vector)
    return ((vector - min_val) / (max_val - min_val)).tolist()

log_merged['env_info_vector_norm'] = log_merged['env_info_vector'].apply(scale_row_manual)

# ======= Final output =======
# log_merged now contains:
# - 'prefix': sequence of activities for each patient
# - 'waiting_time_norm': normalized waiting time
# - 'env_info_vector_norm': normalized environment state vector


## Hàm reward:
reward = alpha * (1 - norm(waiting_time at node t+1)) + (1-alpha) * (1 - avg(norm(env_info_vector at t+1)))
alpha = 0.6

In [2]:
alpha = 0.6

all_states = []
all_actions = []
all_next_states = []
all_rewards = []

for pid in log_merged['patient_id'].unique():
    log_of_case = log_merged[log_merged['patient_id'] == pid].sort_values('start_timestamp')
    
    state = []
    action = []
    next_state = []
    reward = []
    
    for i in range(len(log_of_case) - 1):
        current_row = log_of_case.iloc[i]
        next_row = log_of_case.iloc[i + 1]
        
        # Current state = prefix + env_info_vector
        state.append(current_row['prefix'] + next_row['env_info_vector_norm']) 
        # Action = activity at t+1
        action.append(next_row['activity_name'])
        # Next state = prefix + env_info_vector at t+1
        next_state.append(next_row['prefix'] + next_row['env_info_vector_norm'])
        
        # Reward calculation
        waiting_time_reward = 1 - next_row['waiting_time_norm']  # already normalized
        env_reward = 1 - np.mean(next_row['env_info_vector_norm'])  # normalized vector
        r = alpha * waiting_time_reward + (1 - alpha) * env_reward
        reward.append(r)
    
    # Collect per patient
    all_states.extend(state)
    all_actions.extend(action)
    all_next_states.extend(next_state)
    all_rewards.extend(reward)

In [3]:
data = {
    'state': all_states,
    'action': all_actions,
    'next_state': all_next_states,
    'reward': all_rewards
}

df = pd.DataFrame(data)

In [4]:
df

,state,action,next_state,reward
0,"[Registration, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...",Payment,"[Registration, Payment, 1.0, 0.0, 0.0, 0.0, 0....",0.980952
1,"[Registration, Payment, 1.0, 0.4, 0.0, 0.0, 0....",Get Triage Number,"[Registration, Payment, Get Triage Number, 1.0...",0.973333
2,"[Registration, Payment, Get Triage Number, 1.0...",Measure Vital Signs,"[Registration, Payment, Get Triage Number, Mea...",0.973333
3,"[Registration, Payment, Get Triage Number, Mea...",General Medicine Examination,"[Registration, Payment, Get Triage Number, Mea...",0.961905
4,"[Registration, Payment, Get Triage Number, Mea...",Eye Examination,"[Registration, Payment, Get Triage Number, Mea...",0.971429
...,...,...,...,...
1215,"[Registration, Payment, Get Triage Number, Mea...",DEXA Bone Density Scan,"[Registration, Payment, Get Triage Number, Mea...",0.892063
1216,"[Registration, Payment, Get Triage Number, Mea...",Blood Test,"[Registration, Payment, Get Triage Number, Mea...",0.902646
1217,"[Registration, Payment, Get Triage Number, Mea...",Post-bronchodilator Spirometry,"[Registration, Payment, Get Triage Number, Mea...",0.602398
1218,"[Registration, Payment, Get Triage Number, Mea...",Electrocardiogram (ECG),"[Registration, Payment, Get Triage Number, Mea...",0.955556


In [8]:
with open("../../data/preprocess_data/general_check_up/general_check_up_preprocess.csv", "w") as f:
    df.to_csv(f, index=False)